# Lecture 19: Capstone & Future Trends
### NLP Course 2027

---

## Learning Outcomes
- Synthesize the full NLP landscape from the course
- Understand large language models (LLMs) and prompt engineering
- Build a simple RAG (Retrieval-Augmented Generation) system
- Critically evaluate ethical implications of NLP systems

**Primary References:** *Practical NLP* Ch.12 | *NLP with Transformers* Ch.9-11

## 1. Course Retrospective

### What We Covered

```
Week 1-2:  Foundations
  L1: What is NLP + NLTK intro
  L2: Python text processing
  L3: Corpora + WordNet
  L4: Regex + chunking

Week 3-4:  Classical NLP
  L5: Text classification
  L6: POS tagging + NER
  L7: Information extraction
  L8: NLP pipeline design

Week 5:    Representations
  L9:  TF-IDF, Word2Vec, embeddings
  L10: Deep learning (LSTM, CNN)

Week 6-7:  Transformers
  L11: Attention mechanism
  L12: Hugging Face ecosystem
  L13: Fine-tuning BERT
  L14: Advanced topics (multilingual, LoRA)

Week 8-9:  Applications & Production
  L15: Chatbots + dialogue
  L16: IR, topic modeling, summarization
  L17: Industry domains
  L18: Production deployment

Week 10:   Capstone + Future
  L19: This lecture!
```

## 2. The LLM Era

### Evolution of Language Models
```
2017  Transformer (Attention Is All You Need)
2018  BERT (Google) - bidirectional pretraining
2019  GPT-2 (OpenAI) - 1.5B params, text generation
2020  GPT-3 (OpenAI) - 175B params, few-shot learning
2022  ChatGPT - RLHF, conversational AI
2023  GPT-4, Claude, Llama 2, Mistral
2024  Gemini, Claude 3, Llama 3, Qwen, Phi
2025+ Multimodal, agentic, specialized models
```

### Key Innovations

| Innovation | What it means |
|------------|---------------|
| Instruction tuning | LLMs follow natural language instructions |
| RLHF | Human feedback aligns model behavior |
| Chain-of-thought | Step-by-step reasoning improves accuracy |
| In-context learning | Few examples in prompt teach new tasks |
| Tool use / agents | LLMs call external APIs and tools |

In [1]:
# Prompt engineering techniques
# These work with any LLM API (OpenAI, Anthropic, etc.)

# Zero-shot prompting
zero_shot = '''
Classify the sentiment of this review as POSITIVE, NEGATIVE, or NEUTRAL.
Review: "The delivery was fast but the product quality was disappointing."
Sentiment:
'''

# Few-shot prompting
few_shot = '''
Classify the sentiment of each review.

Review: "Absolutely love this product! Works perfectly."
Sentiment: POSITIVE

Review: "Terrible quality, broke after one day."
Sentiment: NEGATIVE

Review: "Arrived on time. Does what it says."
Sentiment: NEUTRAL

Review: "The delivery was fast but the product quality was disappointing."
Sentiment:
'''

# Chain-of-thought prompting
cot_prompt = '''
Analyze this review step by step, then classify the overall sentiment.

Review: "The delivery was fast but the product quality was disappointing."

Step 1 - Positive aspects:
Step 2 - Negative aspects:
Step 3 - Overall balance:
Final sentiment:
'''

print('=== Zero-shot prompt ===')
print(zero_shot)
print('=== Few-shot prompt ===')
print(few_shot[:200], '...')
print('\n=== Chain-of-thought prompt ===')
print(cot_prompt)

=== Zero-shot prompt ===

Classify the sentiment of this review as POSITIVE, NEGATIVE, or NEUTRAL.
Review: "The delivery was fast but the product quality was disappointing."
Sentiment:

=== Few-shot prompt ===

Classify the sentiment of each review.

Review: "Absolutely love this product! Works perfectly."
Sentiment: POSITIVE

Review: "Terrible quality, broke after one day."
Sentiment: NEGATIVE

Review: "Ar ...

=== Chain-of-thought prompt ===

Analyze this review step by step, then classify the overall sentiment.

Review: "The delivery was fast but the product quality was disappointing."

Step 1 - Positive aspects:
Step 2 - Negative aspects:
Step 3 - Overall balance:
Final sentiment:



In [2]:
# Using an LLM API (concept - replace with your API key)
llm_api_code = '''
import anthropic

client = anthropic.Anthropic(api_key="your-api-key")

# Zero-shot classification
message = client.messages.create(
    model="claude-opus-4-6",
    max_tokens=100,
    messages=[{
        "role": "user",
        "content": """Classify the sentiment of this review as POSITIVE, NEGATIVE, or NEUTRAL.
        Only reply with one word.
        Review: The delivery was fast but the product quality was disappointing."""
    }]
)
print(message.content[0].text)

# Structured extraction
extraction = client.messages.create(
    model="claude-opus-4-6",
    max_tokens=300,
    messages=[{
        "role": "user",
        "content": """Extract entities from this text as JSON:
        {"persons": [], "organizations": [], "locations": []}

        Text: Tim Cook announced Apple\'s new iPhone at the event in Cupertino."""
    }]
)
print(extraction.content[0].text)
'''
print('LLM API usage (replace API key):')
print(llm_api_code)

LLM API usage (replace API key):

import anthropic

client = anthropic.Anthropic(api_key="your-api-key")

# Zero-shot classification
message = client.messages.create(
    model="claude-opus-4-6",
    max_tokens=100,
    messages=[{
        "role": "user",
        "content": """Classify the sentiment of this review as POSITIVE, NEGATIVE, or NEUTRAL.
        Only reply with one word.
        Review: The delivery was fast but the product quality was disappointing."""
    }]
)
print(message.content[0].text)

# Structured extraction
extraction = client.messages.create(
    model="claude-opus-4-6",
    max_tokens=300,
    messages=[{
        "role": "user",
        "content": """Extract entities from this text as JSON:
        {"persons": [], "organizations": [], "locations": []}

        Text: Tim Cook announced Apple's new iPhone at the event in Cupertino."""
    }]
)
print(extraction.content[0].text)



## 3. Retrieval-Augmented Generation (RAG)

RAG combines retrieval (search) with generation (LLM):

```
Question: 'What are the main NLP applications in healthcare?'
       |
       v
[1] Retriever: search knowledge base for relevant docs
       |
       v
[2] Retrieved context: clinical NLP, BioBERT, de-identification...
       |
       v
[3] Generator (LLM): answer based on retrieved context
       |
       v
Answer: 'In healthcare, NLP is used for clinical note analysis,
         de-identification, medical coding, and drug extraction...'
```

### Why RAG?
- LLMs have a knowledge cutoff date
- LLMs can hallucinate facts
- RAG grounds answers in real documents
- No need to retrain the LLM

In [3]:
# Simple RAG pipeline with FAISS + sentence-transformers
# !pip install faiss-cpu sentence-transformers

try:
    from sentence_transformers import SentenceTransformer
    import numpy as np

    # Knowledge base (course notes)
    knowledge_base = [
        'BERT is a bidirectional transformer model trained on masked language modeling and next sentence prediction. It achieves state-of-the-art results on 11 NLP tasks.',
        'Fine-tuning means taking a pretrained model like BERT and training it further on a task-specific labeled dataset. It typically requires only 2-5 epochs.',
        'Named Entity Recognition (NER) is the task of identifying named entities (people, organizations, places) in text and classifying them into predefined categories.',
        'TF-IDF stands for Term Frequency - Inverse Document Frequency. It is a numerical statistic used to reflect the importance of a word in a document.',
        'Word2Vec is a neural network that learns word embeddings by predicting context words. It captures semantic meaning: king - man + woman = queen.',
        'The transformer attention mechanism allows each token to attend to all other tokens in the sequence. This enables capturing long-range dependencies efficiently.',
        'SpaCy is a production-ready NLP library. It provides tokenization, POS tagging, NER, and dependency parsing. It is faster than NLTK for many tasks.',
        'NLTK (Natural Language Toolkit) is an educational NLP library for Python. It provides access to corpora, tokenizers, taggers, and parsers.',
        'Sentiment analysis classifies text as positive, negative, or neutral. Common approaches include Naive Bayes, Logistic Regression, and fine-tuned BERT.',
        'RAG (Retrieval-Augmented Generation) combines a retriever that finds relevant documents with a generator (LLM) that produces answers grounded in those documents.',
        'LoRA (Low-Rank Adaptation) is a parameter-efficient fine-tuning method. It trains low-rank update matrices instead of all model weights, reducing trainable params by 99%.',
        'Tokenization is the process of splitting text into smaller units called tokens. BERT uses WordPiece tokenization which handles out-of-vocabulary words via subwords.',
    ]

    # Encode knowledge base
    encoder = SentenceTransformer('all-MiniLM-L6-v2')
    kb_embeddings = encoder.encode(knowledge_base, show_progress_bar=False)

    def retrieve(query, k=3):
        query_emb = encoder.encode([query])
        from sklearn.metrics.pairwise import cosine_similarity
        sims = cosine_similarity(query_emb, kb_embeddings)[0]
        top_k = np.argsort(sims)[::-1][:k]
        return [(knowledge_base[i], sims[i]) for i in top_k]

    def rag_answer(query):
        docs = retrieve(query, k=3)
        context = '\n'.join(f'- {doc}' for doc, _ in docs)
        prompt = f'Based on the context below, answer the question.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:'
        return prompt, docs

    # Test
    questions = [
        'What is BERT and what is it used for?',
        'How does fine-tuning work?',
        'What is the difference between NLTK and spaCy?',
    ]
    for q in questions:
        prompt, docs = rag_answer(q)
        print(f'Q: {q}')
        print('Retrieved context:')
        for doc, score in docs:
            print(f'  [{score:.3f}] {doc[:80]}...')
        print()

except ImportError:
    print('Install: pip install sentence-transformers faiss-cpu')
    print('RAG pipeline requires an embedding model + a retriever + an LLM')

/opt/miniconda3/envs/nlp-course/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Q: What is BERT and what is it used for?
Retrieved context:
  [0.562] BERT is a bidirectional transformer model trained on masked language modeling an...
  [0.527] Tokenization is the process of splitting text into smaller units called tokens. ...
  [0.316] NLTK (Natural Language Toolkit) is an educational NLP library for Python. It pro...

Q: How does fine-tuning work?
Retrieved context:
  [0.712] Fine-tuning means taking a pretrained model like BERT and training it further on...
  [0.365] LoRA (Low-Rank Adaptation) is a parameter-efficient fine-tuning method. It train...
  [0.246] The transformer attention mechanism allows each token to attend to all other tok...

Q: What is the difference between NLTK and spaCy?
Retrieved context:
  [0.767] SpaCy is a production-ready NLP library. It provides tokenization, POS tagging, ...
  [0.586] NLTK (Natural Language Toolkit) is an educational NLP library for Python. It pro...
  [0.377] Tokenization is the process of splitting text into smaller

In [4]:
# AI Agents: LLMs that use tools
agent_code = '''
class SimpleNLPAgent:
    """
    A simple agent that uses tools to answer NLP questions.
    In practice, use LangChain or LlamaIndex.
    """
    def __init__(self, llm):
        self.llm = llm
        self.tools = {
            'sentiment': self.run_sentiment,
            'ner': self.run_ner,
            'summarize': self.run_summarize,
        }

    def run_sentiment(self, text):
        from transformers import pipeline
        pipe = pipeline("text-classification")
        return pipe(text)[0]

    def run_ner(self, text):
        from transformers import pipeline
        pipe = pipeline("ner", aggregation_strategy="simple")
        return pipe(text)

    def run_summarize(self, text):
        from transformers import pipeline
        pipe = pipeline("summarization", max_length=50)
        return pipe(text)[0]["summary_text"]

    def act(self, task):
        """
        1. Decide which tool to use (via LLM)
        2. Run the tool
        3. Return the result
        """
        # In real agents: LLM decides the tool
        # Here: simple keyword matching
        task_lower = task.lower()
        if "sentiment" in task_lower:
            text = task.split(":")[-1].strip()
            return f"Tool: sentiment\nResult: {self.run_sentiment(text)}"
        elif "entities" in task_lower or "ner" in task_lower:
            text = task.split(":")[-1].strip()
            return f"Tool: ner\nResult: {self.run_ner(text)}"
        elif "summarize" in task_lower or "summary" in task_lower:
            text = task.split(":")[-1].strip()
            return f"Tool: summarize\nResult: {self.run_summarize(text)}"
        return "Tool: none\nCannot determine the right tool for this task."
'''
print('AI Agent skeleton:')
print(agent_code)

AI Agent skeleton:

class SimpleNLPAgent:
    """
    A simple agent that uses tools to answer NLP questions.
    In practice, use LangChain or LlamaIndex.
    """
    def __init__(self, llm):
        self.llm = llm
        self.tools = {
            'sentiment': self.run_sentiment,
            'ner': self.run_ner,
            'summarize': self.run_summarize,
        }

    def run_sentiment(self, text):
        from transformers import pipeline
        pipe = pipeline("text-classification")
        return pipe(text)[0]

    def run_ner(self, text):
        from transformers import pipeline
        pipe = pipeline("ner", aggregation_strategy="simple")
        return pipe(text)

    def run_summarize(self, text):
        from transformers import pipeline
        pipe = pipeline("summarization", max_length=50)
        return pipe(text)[0]["summary_text"]

    def act(self, task):
        """
        1. Decide which tool to use (via LLM)
        2. Run the tool
        3. Return the resul

## 4. Ethical Considerations

With great power comes great responsibility. NLP systems can cause real harm.

### Bias in NLP Models

| Source of Bias | Example | Harm |
|---------------|---------|------|
| Training data | Web text over-represents English, Western views | Multilingual underperformance |
| Label bias | Annotators disagree on what is 'toxic' | Inconsistent moderation |
| Word embeddings | 'doctor' closer to 'man', 'nurse' to 'woman' | Reinforces stereotypes |
| Class imbalance | More positive reviews than negative | Biased classifier |

### Privacy Concerns
- LLMs trained on web text may memorize personal information
- Clinical NLP processes sensitive health data (HIPAA)
- Named entity recognition can de-anonymize supposedly private data

### Environmental Impact
- Training GPT-3: ~552 tonnes CO2 (equivalent to 300 NYC-to-SF flights)
- Inference at scale: significant ongoing energy use
- Optimization matters: DistilBERT saves 60% energy vs BERT

In [5]:
# Bias detection in word embeddings
from gensim.models import Word2Vec
from nltk.corpus import gutenberg
from nltk.tokenize import word_tokenize, sent_tokenize
import nltk

nltk.download('gutenberg', quiet=True)
nltk.download('punkt', quiet=True)

# Train Word2Vec on Gutenberg corpus
print('Loading corpus...')
sentences = [word_tokenize(s.lower())
             for text_id in gutenberg.fileids()
             for s in sent_tokenize(gutenberg.raw(text_id))]
model = Word2Vec(sentences, vector_size=100, window=5, min_count=10,
                 workers=4, epochs=10, seed=42)
print(f'Vocabulary: {len(model.wv)} words')

# Test for gender bias in professions
professions = ['doctor', 'nurse', 'engineer', 'teacher', 'lawyer',
               'scientist', 'secretary', 'professor']
print('\nGender bias in profession embeddings:')
print('(+ = closer to male concepts, - = closer to female concepts)')
print('-' * 50)
for prof in professions:
    try:
        sim_man = model.wv.similarity(prof, 'man')
        sim_woman = model.wv.similarity(prof, 'woman')
        bias = sim_man - sim_woman
        direction = 'male' if bias > 0 else 'female'
        print(f'  {prof:12s}: bias={bias:+.4f}  ({direction})')
    except KeyError:
        print(f'  {prof}: not in vocabulary')

Loading corpus...
Vocabulary: 11026 words

Gender bias in profession embeddings:
(+ = closer to male concepts, - = closer to female concepts)
--------------------------------------------------
  doctor      : bias=+0.0466  (male)
  nurse       : bias=-0.2677  (female)
  engineer: not in vocabulary
  teacher     : bias=-0.0541  (female)
  lawyer      : bias=+0.0040  (male)
  scientist: not in vocabulary
  secretary   : bias=+0.0039  (male)
  professor   : bias=+0.0097  (male)


In [6]:
# Fairness evaluation across demographic groups
from transformers import pipeline
import numpy as np

sentiment = pipeline('text-classification',
                     model='distilbert-base-uncased-finetuned-sst-2-english')

# Test for sentiment consistency across demographic groups
# (swap demographic terms; predictions should be same)

def test_demographic_parity(template, groups, expected='POSITIVE'):
    """
    Test if model gives same predictions when swapping demographic terms.
    Template: 'The {group} person did a great job.'
    """
    results = {}
    for group in groups:
        text = template.format(group=group)
        result = sentiment(text)[0]
        results[group] = {'label': result['label'], 'score': result['score']}
    return results

template = 'The {group} employee did an excellent job on this project.'
groups = ['male', 'female', 'young', 'elderly', 'Christian', 'Muslim', 'Asian', 'Black', 'White']

print(f'Template: {template}')
print('Sentiment by demographic group:')
results = test_demographic_parity(template, groups)
for group, result in results.items():
    flag = '' if result['label'] == 'POSITIVE' else '<<BIAS>>'
    print(f'  {group:12s}: {result["label"]:10s} ({result["score"]:.3f}) {flag}')

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Template: The {group} employee did an excellent job on this project.
Sentiment by demographic group:
  male        : POSITIVE   (1.000) 
  female      : POSITIVE   (1.000) 
  young       : POSITIVE   (1.000) 
  elderly     : POSITIVE   (0.999) 
  Christian   : POSITIVE   (1.000) 
  Muslim      : POSITIVE   (0.999) 
  Asian       : POSITIVE   (1.000) 
  Black       : POSITIVE   (1.000) 
  White       : POSITIVE   (1.000) 


## 5. Future Trends

### Near-term (1-3 years)
- **Multimodal LLMs**: text + image + audio + video
- **Smaller, efficient models**: Phi-3, Gemma, Llama 3 (runs on laptop)
- **Better reasoning**: chain-of-thought, process reward models
- **Agentic AI**: autonomous task completion with tools

### Medium-term (3-5 years)
- **Personalized models**: fine-tuned to individual users
- **Neuro-symbolic AI**: LLMs + structured reasoning
- **Better factuality**: reduced hallucination
- **Responsible AI frameworks**: regulation (EU AI Act)

### Open Research Problems
- **Long context**: handling million-token contexts efficiently
- **Continual learning**: learning without forgetting
- **True language understanding**: grounding in world knowledge
- **Interpretability**: understanding why models make predictions
- **Alignment**: ensuring models do what we want

## 6. Capstone Project Guidelines

### Project Requirements

**Option A: End-to-End NLP Application**
- Choose a real-world problem (sentiment, classification, IE, chatbot, etc.)
- Collect or use a public dataset
- Build and evaluate a complete pipeline
- Deploy as a simple API or demo app
- Write a 5-page report

**Option B: Research Extension**
- Pick a technique from the course
- Run systematic experiments (e.g., compare fine-tuning strategies)
- Reproduce and extend a published paper
- Write a 8-page report

### Evaluation Rubric
| Criterion | Weight | Description |
|-----------|--------|-------------|
| Problem statement | 15% | Clear motivation and scope |
| Data & preprocessing | 20% | Quality pipeline, right choices |
| Model & methods | 25% | Appropriate model, justified choice |
| Evaluation | 20% | Correct metrics, error analysis |
| Presentation | 10% | Clear communication |
| Code quality | 10% | Clean, reproducible, documented |

In [7]:
# Capstone project starter template

capstone_template = '''
## Project: [Your Project Title]

### 1. Problem Statement
# What NLP problem are you solving?
# Who benefits from this solution?
# What is the baseline and target performance?

### 2. Data
# Source: [dataset name, URL]
# Size: [number of examples, train/val/test split]
# Format: [text + labels, text pairs, etc.]

### 3. NLP Pipeline
pipeline_steps = {
    "preprocessing": [
        "tokenization",     # which tokenizer?
        "cleaning",         # what noise to remove?
        "normalization",    # stem/lemmatize?
    ],
    "representation": "TF-IDF / Word2Vec / BERT embeddings",
    "model": "Naive Bayes / LSTM / Fine-tuned DistilBERT",
    "evaluation": "accuracy / F1 / BLEU / ROUGE",
}

### 4. Experiments
experiments = [
    {"name": "Baseline", "model": "TF-IDF + LR", "f1": None},
    {"name": "Deep",     "model": "LSTM",         "f1": None},
    {"name": "BERT",     "model": "DistilBERT",   "f1": None},
]

### 5. Results
results_template = """
| Model | Precision | Recall | F1 | Notes |
|-------|-----------|--------|-------|-------|
| Baseline | | | | |
| LSTM | | | | |
| DistilBERT | | | | |
"""

### 6. Error Analysis
# What types of errors does the best model make?
# Confusion matrix, failure cases, out-of-domain examples

### 7. Deployment
# REST API with FastAPI
# Demo with Gradio or Streamlit
'''
print('Capstone project template:')
print(capstone_template)

Capstone project template:

## Project: [Your Project Title]

### 1. Problem Statement
# What NLP problem are you solving?
# Who benefits from this solution?
# What is the baseline and target performance?

### 2. Data
# Source: [dataset name, URL]
# Size: [number of examples, train/val/test split]
# Format: [text + labels, text pairs, etc.]

### 3. NLP Pipeline
pipeline_steps = {
    "preprocessing": [
        "tokenization",     # which tokenizer?
        "cleaning",         # what noise to remove?
        "normalization",    # stem/lemmatize?
    ],
    "representation": "TF-IDF / Word2Vec / BERT embeddings",
    "model": "Naive Bayes / LSTM / Fine-tuned DistilBERT",
    "evaluation": "accuracy / F1 / BLEU / ROUGE",
}

### 4. Experiments
experiments = [
    {"name": "Baseline", "model": "TF-IDF + LR", "f1": None},
    {"name": "Deep",     "model": "LSTM",         "f1": None},
    {"name": "BERT",     "model": "DistilBERT",   "f1": None},
]

### 5. Results
results_template = """
| Mod

## Course Summary

### The NLP Toolkit You Now Have

```
Foundations:       NLTK, regex, corpus analysis, WordNet
Classical ML:      TF-IDF, Naive Bayes, SVM, Logistic Regression
Sequence models:   POS taggers, NER, chunking, IE pipelines
Embeddings:        Word2Vec, GloVe, FastText
Deep learning:     LSTM, CNN, attention
Transformers:      BERT, DistilBERT, RoBERTa, XLM-R
Fine-tuning:       Hugging Face Trainer, LoRA, adapters
Applications:      Chatbots, IR, topic modeling, summarization
Domains:           Social media, healthcare, legal, e-commerce
Production:        FastAPI, ONNX, monitoring, MLflow
LLMs:              Prompt engineering, RAG, agents
Ethics:            Bias detection, privacy, fairness
```

### Key Books
1. *Natural Language Processing with Python* (Bird, Klein, Loper) -- free: nltk.org/book
2. *Natural Language Processing with Transformers* (Tunstall et al.) -- practical HF guide
3. *Practical Natural Language Processing* (Vajjala et al.) -- industry focus

**Congratulations on completing the NLP Course 2027!**

---
*Book references: Practical NLP Ch.12 | NLP with Transformers Ch.9-11*

## Practice Exercises

See **`Lecture-19-Homework.ipynb`** for the practice exercises accompanying this lecture.

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**